<a href="https://colab.research.google.com/github/willyd520/Math311_Linear_Algebra/blob/main/Copy_of_Sports_Teams_Rankings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load the preliminary code

In [1]:
import matplotlib.pyplot as plt
from sympy import Poly
import sympy as sym
import numpy as np
from numpy import c_

# We'll specify that x and t are variables:
x, t = sym.symbols('x, t')

ll = "loaded"

In [2]:
lll = " correctly!"

def rref(matrix):
    A = np.array(matrix, dtype=np.float64)

    r = 0 #row
    c = 0 #column

    (maxr, maxc) = A.shape

    prev_row_pivot = -1   # The previous row with a leading term:
    tmp_pivot_row = -1
    this_pivot_row = -1

    for c in range(maxc):  # go through each column
      tmp_pivot_row = -1  # reset the tmp pivot row
      this_pivot_row = -1  # reset this pivot row
      for r in range(maxr):  # Find the leading term in this column below prev_row_pivot
        if ((r > prev_row_pivot) and (A[r][c] != 0.0)):
          tmp_pivot_row = r  # find the new pivot row
      if (tmp_pivot_row > -1):  # we found a candidate for the leading term:
        #swap row this_pivot_row with (prev_row_pivot + 1)
        A[[tmp_pivot_row, prev_row_pivot+1]] = A[[prev_row_pivot + 1, tmp_pivot_row]]
        #set this_pivot_row to prev_row_pivot+1
        this_pivot_row = prev_row_pivot + 1
        #Now, normalize the row based on the pivot value:
        A[ this_pivot_row ] = A[ this_pivot_row ] / A[ this_pivot_row ][c]
        #Then, use this row to force 0's above and below the pivot entry:
        #subtract multiples of of the this_pivot_row row from the others
        for r_ in range(len(A)):
            if r_ != this_pivot_row:
                A[r_] = A[r_] - A[this_pivot_row] * A[r_][c] / A[this_pivot_row][c]
        prev_row_pivot = this_pivot_row
    return A


def is_singular(matrix):
  singular = True
  notsingular = False

  (mr, mc) = matrix.shape

  return_value = notsingular
  if (mc - np.linalg.linalg.matrix_rank(matrix) > 0) :
    return_value = singular
  return return_value




print(">> The page ", ll, lll)

>> The page  loaded  correctly!


# Ranking Teams based on games

From our packet, we saw there were five games:

Home  | Away
------|------
Winona State 16 | Sioux Falls 15
Winona State 34 | Augustana 20
Sioux Falls 51  | Augustana 13
Duluth 28       |  Augustana 17
Winona State 17 | Duluth 37

Let's encode this system into a matrix.

The unknown rankings are $r_w, r_s, r_a, r_d$.

The system looks like $M \vec{x} = \vec{b}$, or

$$
\left( \begin{array}{cccc} 1 & -1 & 0 & 0 \\ 1 & 0 & -1 & 0 \\ 0 & 1 & -1 & 0 \\ 0 & 0 & -1 & 1 \\ -1 & 0  & 0 & 1 \\ \end{array} \right) \, \left( \begin{array}{c} r_w \\ r_s \\ r_a \\ r_d \\ \end{array} \right) = \left( \begin{array}{c} 1 \\ 14 \\ 38 \\ 11 \\ 20 \\ \end{array} \right)$$

We'll solve this system using least squares
$$M^T I M \vec{x} = M^T I \vec{b} $$

$$ \vec{x} = \left( M^T I M  \right)^{-1} M^T I \vec{b} $$






In [ ]:
# create the matrix:
M = np.matrix([[1, -1, 0, 0],
               [1, 0, -1, 0],
               [0, 1, -1, 0],
               [0, 0, -1, 1],
               [-1, 0, 0, 1]])


# create the vector for the right hand side:
pts = np.matrix([[1], [14], [38], [11], [20]])

#Make sure the system looks appropriate:
print(np.c_[M,pts])

[[ 1 -1  0  0  1]
 [ 1  0 -1  0 14]
 [ 0  1 -1  0 38]
 [ 0  0 -1  1 11]
 [-1  0  0  1 20]]


In [ ]:
# Generate the transpose of M:
MT = M.transpose()

print("The Transpose of M is:")
print(MT)

# and find MT*M and MT*pts:
print("\n")
print("MT * M is: ")
MTM = MT * M
print(MTM)

print("\n")
print("MT * pts is:")
MTpts = MT * pts
print(MTpts)


The Transpose of M is:
[[ 1  1  0  0 -1]
 [-1  0  1  0  0]
 [ 0 -1 -1 -1  0]
 [ 0  0  0  1  1]]


MT * M is: 
[[ 3 -1 -1 -1]
 [-1  2 -1  0]
 [-1 -1  3 -1]
 [-1  0 -1  2]]


MT * pts is:
[[ -5]
 [ 37]
 [-63]
 [ 31]]


If $M^T M$ has an inverse, then we can use it.  But, if $M^T M$ is singular, then it doesn't have an inverse.  In this situation, we need to replace every entry in one row (say the bottom row) of $M^T M$ and $M^T \vec{pts}$ with 1's.  (This knocks the system out of *not* having an inverse, so it will now have an inverse.)

In [ ]:
#  Check if MTM is singular
is_singular(MT*M)

True

In [ ]:
# If MTM is singular, then adjust one row of MTM and MT pts to be just 1's:

#MTM[3] is the entire 4th row of MTM,
#so MTM[3] = 1 will put 1's in MTM's 4th row.

MTM[3] = 1
MTpts[3] = 1

print("MTM is now: ")
print(MTM)

print("and MTpts is now:")
print(MTpts)

print("Is our modified MTM singular?")
is_singular(MTM)

MTM is now: 
[[ 3 -1 -1 -1]
 [-1  2 -1  0]
 [-1 -1  3 -1]
 [ 1  1  1  1]]
and MTpts is now:
[[ -5]
 [ 37]
 [-63]
 [  1]]
Is our modified MTM singular?


False

In [ ]:
#Find our solution using the inverse of MTM:

#First, find the inverse of MTM:
MTMI = np.linalg.inv(MTM)

print("The inverse of MTM is:")
print(MTMI)

#Now, use the inverse of MTM to solve for the rankings:

print("\n")
print("The solution are rankings:")
MTMI * MTpts

The inverse of MTM is:
[[ 2.50000000e-01  0.00000000e+00  1.85037171e-17  2.50000000e-01]
 [ 1.25000000e-01  5.00000000e-01  1.25000000e-01  2.50000000e-01]
 [ 0.00000000e+00  0.00000000e+00  2.50000000e-01  2.50000000e-01]
 [-3.75000000e-01 -5.00000000e-01 -3.75000000e-01  2.50000000e-01]]


The solution are rankings:


matrix([[ -1.  ],
        [ 10.25],
        [-15.5 ],
        [  7.25]])

#  Game Data  - Fall Sports

Find the game data for the four teams in the NFC North and NFC South.

The teams are:

* **NFC North:**  Green Bay Packers, Chicago Bears, Detroit Lions, Minnesota Vikings
* **NFC South:**  Atlanta Falcons, Carolina Panthers, New Orleans Saints, Tampa Bay Buccaneers


It may be helpful to do a google search to find the data.

Be sure to create your matrix on the handout, then enter the data for the system below.

# Game Data - Spring Sports

Find the game data for the seven teams in the Western Division for Men's Big Ten Basketball.

**The teams:**  Wisconsin, Minnesota, Iowa, Illinois, Nebraska, Purdue, and Northewstern.

I've provided **most** of the data, but you have to do a Google search to find the scores for the last two teams.

Be sure to create your matrix on the handout, then enter the data for the system below.


In [27]:
# create the matrix:
M = np.matrix([[0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,1,0,0,],
[0,0,0,0,0,0,0,0,0,0,0,0,-1,0,1,0,0,0,],
[-1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,],
[0,0,0,-1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,],
[0,0,0,0,0,-1,0,1,0,0,0,0,0,0,0,0,0,0,],
[0,0,1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,],
[0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,1,],
[0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1,0,0,],
[0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,-1,0,],
[0,1,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,],
[0,0,0,0,-1,0,1,0,0,0,0,0,0,0,0,0,0,0,],
[0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,1,],
[0,0,-1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,],
[0,0,0,-1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,],
[-1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,-1,0,],
[0,0,0,0,0,0,0,0,1,0,0,-1,0,0,0,0,0,0,],
[0,1,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,],
[0,1,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,],
[1,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,],
[0,0,0,0,0,0,0,0,0,0,-1,0,0,0,1,0,0,0,],
[0,0,0,-1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,],
[0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,1,0,0,],
[0,0,0,0,0,0,-1,0,0,0,0,0,1,0,0,0,0,0,],
[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,],
[0,0,0,0,0,-1,0,0,1,0,0,0,0,0,0,0,0,0,],
[0,-1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,],
[0,0,0,-1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,],
[0,0,0,0,1,0,0,0,0,0,0,-1,0,0,0,0,0,0,],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,-1,],
[0,0,0,0,0,0,0,-1,0,0,1,0,0,0,0,0,0,0,],
[0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,1,0,]])


# create the vector for the right hand side:
pts = np.matrix([[7],[14],[23],[51],[18],[1],[4],[14],[3],[6],[6],[28],[17],[17],[7],[4],[21],[32],[3],[1],[2],[29],[8],[28],[17],[3],[4],[7],[3],[10],[3],[12]])

#Make sure the system looks appropriate:
print(np.c_[M,pts])

[[ 0  0  0  0  0  0  0  0  0  0  0  0  0 -1  0  1  0  0  7]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 -1  0  1  0  0  0 14]
 [-1  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0 23]
 [ 0  0  0 -1  0  0  0  0  0  0  0  0  0  0  1  0  0  0 51]
 [ 0  0  0  0  0 -1  0  1  0  0  0  0  0  0  0  0  0  0 18]
 [ 0  0  1  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  1]
 [ 0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  1  4]
 [ 0  0  0  0  0  0  0  0  0  0  1  0  0  0  0 -1  0  0 14]
 [ 0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0 -1  0  3]
 [ 0  1  0  0  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  6]
 [ 0  0  0  0 -1  0  1  0  0  0  0  0  0  0  0  0  0  0  6]
 [ 0  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  1 28]
 [ 0  0 -1  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0 17]
 [ 0  0  0 -1  0  0  0  1  0  0  0  0  0  0  0  0  0  0 17]
 [-1  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  7]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  4]
 [ 0  0  0  0  0  0  0  0  1  0  0 -1  0

In [29]:
# Generate the transpose of M:
MT = M.transpose()

print("The Transpose of M is:")
print(MT)

# and find MT*M and MT*pts:
print("\n")
print("MT * M is: ")
MTM = MT * M
print(MTM)

print("\n")
print("MT * pts is:")
MTpts = MT * pts
print(MTpts)


The Transpose of M is:
[[ 0  0 -1  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  1  0  0  0  0
   0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  1  1  0  0  0  0  0
   0  0 -1  0  0  0  0  0]
 [ 0  0  0  0  0  1  0  0  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0
   1  0  0  0  0  0  0  0]
 [ 0  0  0 -1  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0 -1  0  0
   0  0  0 -1  0  0  0  0]
 [ 0  0  0  0  0 -1  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  1  0  0  0]
 [ 0  0  0  0 -1  0  0  0  0  0  0 -1  0  0  0  0  0  0  0 -1  0  0  0  0
   0 -1  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  1  0  1  0  0  0  0  0  0  0  0  0  0  0  0 -1
   0  0  0  0  0  0  0  0]
 [ 0  0  0  0  1  0  0  0  0  0  0  0  0  1  0  0  0  0 -1  0  0  0  0  0
   0  0  0  0  0  0 -1  0]
 [ 0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  1  0  0  0  0  0  0  0
   0  1  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  1 

In [31]:
#  Check if MTM is singular
is_singular(MT*M)

/tmp/ipykernel_927/4116023008.py:2: DeprecationWarning: The numpy.linalg.linalg has been made private and renamed to numpy.linalg._linalg. All public functions exported by it are available from numpy.linalg. Please use numpy.linalg.matrix_rank instead.
  is_singular(MT*M)


True

In [33]:
# Careful - you might not need to do this!
# Be sure to check whether MTM is singular on the previous line.

# If MTM is singular, then adjust one row of MTM and MT pts to be just 1's:

#MTM[3] is the entire 4th row of MTM,
#so MTM[3] = 1 will put 1's in MTM's 4th row.

MTM[17] = 1
MTpts[17] = 1

print("MTM is now: ")
print(MTM)

print("and MTpts is now:")
print(MTpts)

print("Is our modified MTM singular?")
is_singular(MTM)

MTM is now: 
[[ 3  0  0  0  0 -1  0  0  0 -1  0  0 -1  0  0  0  0  0]
 [ 0  4  0  0  0  0  0 -1 -1 -1  0  0  0 -1  0  0  0  0]
 [ 0  0  3  0 -1  0  0  0  0 -1  0  0  0  0  0  0 -1  0]
 [ 0  0  0  4  0  0  0 -1  0 -1  0  0  0 -1 -1  0  0  0]
 [ 0  0 -1  0  3  0 -1  0  0  0  0 -1  0  0  0  0  0  0]
 [-1  0  0  0  0  4  0 -1 -1  0  0  0  0  0  0  0  0 -1]
 [ 0  0  0  0 -1  0  3  0  0  0  0  0 -1  0  0  0 -1  0]
 [ 0 -1  0 -1  0 -1  0  4  0  0 -1  0  0  0  0  0  0  0]
 [ 0 -1  0  0  0 -1  0  0  3  0  0 -1  0  0  0  0  0  0]
 [-1 -1 -1 -1  0  0  0  0  0  4  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 -1  0  0  3  0  0  0 -1 -1  0  0]
 [ 0  0  0  0 -1  0  0  0 -1  0  0  4  0  0  0 -1  0 -1]
 [-1  0  0  0  0  0 -1  0  0  0  0  0  4  0 -1  0 -1  0]
 [ 0 -1  0 -1  0  0  0  0  0  0  0  0  0  3  0 -1  0  0]
 [ 0  0  0 -1  0  0  0  0  0  0 -1  0 -1  0  4  0  0 -1]
 [ 0  0  0  0  0  0  0  0  0  0 -1 -1  0 -1  0  4 -1  0]
 [ 0  0 -1  0  0  0 -1  0  0  0  0  0 -1  0  0 -1  4  0]
 [ 1  1  1  1  1  

/tmp/ipykernel_927/386316071.py:19: DeprecationWarning: The numpy.linalg.linalg has been made private and renamed to numpy.linalg._linalg. All public functions exported by it are available from numpy.linalg. Please use numpy.linalg.matrix_rank instead.
  is_singular(MTM)


False

In [34]:
#Find our solution using the inverse of MTM:

#First, find the inverse of MTM:
MTMI = np.linalg.inv(MTM)

print("The inverse of MTM is:")
print(MTMI)

#Now, use the inverse of MTM to solve for the rankings:

print("\n")
print("The solution are rankings:")
MTMI * MTpts

The inverse of MTM is:
[[ 4.00926383e-01 -1.61739289e-04  6.05086777e-05 -2.51652935e-04
  -2.86755863e-02  8.62998975e-02 -9.75460178e-05 -2.20716707e-04
   4.94788239e-05  8.62544861e-02 -3.12138165e-02 -3.04341662e-02
   8.57803220e-02 -3.11746498e-02 -3.10175760e-04 -3.75550015e-02
  -1.84181811e-03  5.55555556e-02]
 [ 1.61123572e-02  3.44710961e-01 -1.19910850e-02  6.61388251e-02
  -3.66906021e-02  4.52674465e-02 -5.23519418e-02  1.05763109e-01
   1.14749876e-01  8.98538757e-02  2.24907573e-02  9.82677606e-03
  -3.12286950e-02  1.24032539e-01  4.61332952e-04  1.68033859e-02
  -3.35809729e-02  5.55555556e-02]
 [ 4.90035455e-02  2.06778554e-02  4.80767961e-01  2.21983893e-02
   2.00740466e-01 -1.80738186e-03  1.29377120e-01 -4.32644736e-03
   3.64892996e-03  1.29273049e-01 -2.81909666e-03  4.76318719e-02
   7.51005250e-02  9.67144550e-03  9.73106550e-03  4.16936474e-02
   1.67845925e-01  5.55555556e-02]
 [ 4.48207870e-03  5.45984602e-02 -2.20109159e-02  3.24384324e-01
  -5.23137150e

matrix([[ -7.25095564],
        [ 12.27822083],
        [  1.33105097],
        [-18.93812953],
        [ -6.88335352],
        [ -9.6143926 ],
        [ -9.78513824],
        [  6.47737215],
        [  4.15595165],
        [ 15.10504666],
        [ 13.18378989],
        [ -8.19597328],
        [  1.75647902],
        [-13.62548713],
        [ 20.29055021],
        [ -2.21655269],
        [ -5.22854023],
        [  8.16006145]])